In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt
/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json
/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt


In [ ]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 48.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/558.3 kB ? eta -:--:--

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_59/3926598702.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 48.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/558.3 kB ? eta -:--:--

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

# LOAD DOUCMENTS 

In [7]:
from langchain_community.document_loaders import TextLoader
spec = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt").load()

interface = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt").load()

rtl = TextLoader("/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json").load()

documents = spec + interface + rtl

print(documents)


[Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/spec.txt'}, page_content='Counter Specification\n\nThe counter increments on every rising edge of the clock.\n\nMaximum value = 15.\n\nAfter reaching 15, the counter wraps back to zero.\n\nReset initializes the counter to zero.'), Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/interface.txt'}, page_content='Inputs\n\nclk\n\nrst\n\nOutputs\n\ncount[3:0]'), Document(metadata={'source': '/kaggle/input/datasets/mahasaad12/verification-documents/rtl_summary.json'}, page_content='{\n    "module":"counter",\n\n    "inputs":[\n        "clk",\n        "rst"\n    ],\n\n    "outputs":[\n        "count"\n    ],\n\n    "registers":[\n        "count"\n    ],\n\n    "always_blocks":1\n}')]


# SPLITT LOAD EMBEDDING MODEL

In [8]:


text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(chunks, embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Create a Retriever (TEST RAG IT SELF)

In [9]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

In [10]:
question = "What happens after count reaches 15?"

In [11]:
retrieved_docs = retriever.invoke(question)

In [12]:
for i, doc in enumerate(retrieved_docs):
    print("=" * 50)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
Counter Specification

The counter increments on every rising edge of the clock.

Maximum value = 15.

After reaching 15, the counter wraps back to zero.

Reset initializes the counter to zero.
Document 2
Inputs

clk

rst

Outputs

count[3:0]
Document 3
{
    "module":"counter",

    "inputs":[
        "clk",
        "rst"
    ],

    "outputs":[
        "count"
    ],

    "registers":[
        "count"
    ],

    "always_blocks":1
}


In [13]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [14]:
print(context)

Counter Specification

The counter increments on every rising edge of the clock.

Maximum value = 15.

After reaching 15, the counter wraps back to zero.

Reset initializes the counter to zero.

Inputs

clk

rst

Outputs

count[3:0]

{
    "module":"counter",

    "inputs":[
        "clk",
        "rst"
    ],

    "outputs":[
        "count"
    ],

    "registers":[
        "count"
    ],

    "always_blocks":1
}


# coverage Test 